# Selección de tipo de entidad

Ejecuta un LLM (LLama-3.1-8B-Instruct) con CoT para determinar cual de los múltiples tipos de entidad extraidos de wikidata, describen mejor a una entidad.

In [1]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from tqdm import tqdm

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["HF_TOKEN"] = "key"

assert torch.backends.mps.is_available()
device = torch.device("mps")

/Users/diegolarraguibel/Desktop/Semestre 2025-2/ia generativa/Proyecto-IA-Gen/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
OUTPUT_DIR = os.path.join("../Datasets", "entity_types_clean")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
context = """Tu tarea consiste en seleccionar, para cada "entidad", la mejor etiqueta dentro de las opciones en "superclase" que la describa de forma más simple, clara y general.  

Debes elegir **una sola palabra o frase corta** que capture de manera eficiente la naturaleza de la entidad, evitando redundancias o términos demasiado específicos.  

---

### MODO DE RAZONAMIENTO (ejemplo a seguir)
1. Analiza qué tipo de cosa es la entidad (¿persona, lugar, organización, concepto?).
2. Revisa las superclases y determina cuál describe de forma más directa, simple y general.
3. Si varias opciones son posibles, prefiere:
   - la más general y común;
   - la que se entiende por sí sola sin contexto;
   - la que sería más útil como etiqueta en un grafo de conocimiento.
   - Si la etiqueta correcta es un substring de una opción, puedes elegirla (e.g. Ciudad de Estados Unidos → Ciudad).
4. Evita términos demasiado específicos, técnicos o redundantes. También evita describir algo por su propio nombre.

---

### EJEMPLOS CON RAZONAMIENTO

**Ejemplo 1**
entidad: Reino Unido  
superclase: país, país insular, estado soberano, poder colonial  
razonamiento:  
Reino Unido es un estado compuesto que cumple con las características de un país soberano. “País insular” y “poder colonial” son descripciones históricas o geográficas, pero “país” es la etiqueta más simple y general.  
output: país  

**Ejemplo 2**
entidad: Rafaela  
superclase: municipio, asentamiento, municipio de Argentina, ciudad de Argentina  
razonamiento:  
Rafaela es una localidad urbana dentro de Argentina. “Asentamiento” y “municipio” son más generales, pero “ciudad de Argentina” es la forma más específica y natural que la describe sin redundancia. Sin embargo, “ciudad” es suficiente y más general, lo que permite generar mejores clases. 
output: ciudad

**Ejemplo 3**
entidad: empresario  
superclase: profesión, persona jurídica, ocupación, concepto económico, business and administration professionals  
razonamiento:  
“Empresario” se refiere a una persona que ejerce una actividad económica. No es una persona jurídica, sino una ocupación o rol laboral. “Ocupación” es la etiqueta más general y adecuada.  
output: ocupación  

**Ejemplo 4**
entidad: ingeniero  
superclase: profesión, cargo, trabajador  
razonamiento:  
“Profesión” describe directamente lo que es ser ingeniero, mientras que “cargo” o “trabajador” son categorías más amplias.  
output: profesión  

**Ejemplo 5**
entidad: pintor  
superclase: profesión, Q778000, trabajador de la construcción, menestral  
razonamiento:  
“Pintor” puede ser artístico o técnico, pero en ambos casos es una profesión. “Trabajador de la construcción” es un subconjunto y “menestral” es arcaico.  
output: profesión 
---

### NUEVO CASO

Ahora aplica el mismo razonamiento anterior, pero **sin mostrar el razonamiento**, solo entrega el resultado final.  

entidad: {{entidad}}  
superclase: {{lista_de_superclases}}  

output:
""".strip()

In [4]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

tok.padding_side = "left"
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to(device)

if model.generation_config.pad_token_id is None and tok.pad_token_id is None:
    model.generation_config.pad_token_id = tok.eos_token_id

PAD_ID = tok.pad_token_id
EOS_ID = tok.eos_token_id

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.50s/it]


In [5]:
SYSTEM_MSG = {"role": "system", "content": context}
prefix_ids = tok.apply_chat_template(
    [SYSTEM_MSG],
    tokenize=True,
    add_generation_prompt=False,
    return_tensors="pt"
).to(device)

In [6]:
def build_user_ids(entity: str, superclasses: list[str]) -> tuple[torch.Tensor, torch.Tensor]:
    """Crea input_ids y attention_mask concatenando el prefijo (system) + turno del usuario."""
    sc_str = ", ".join(superclasses)
    user_msg = {"role": "user", "content": f"entidad: {entity}\nsuperclase: {sc_str}\n\noutput:"}
    user_ids = tok.apply_chat_template(
        [user_msg],
        tokenize=True,
        add_generation_prompt=True,   # añade el turno de assistant
        return_tensors="pt"
    ).to(device)
    input_ids = torch.cat([prefix_ids, user_ids], dim=-1)
    attention_mask = (input_ids != PAD_ID)
    return input_ids, attention_mask

@torch.inference_mode()
def generate_label(entity: str, superclasses: list[str],
                   max_new_tokens: int = 8,
                   do_sample: bool = False) -> str:
    """
    Genera la etiqueta final para una entidad dada su lista de superclases.
    Sin batching. Razonamiento oculto por prompt. Respuesta: una etiqueta.
    """
    input_ids, attention_mask = build_user_ids(entity, superclasses)

    gen_out = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,          # False = greedy (más rápido/estable)
        eos_token_id=EOS_ID,
        pad_token_id=PAD_ID,
        use_cache=True
    )

    gen_ids = gen_out[0, input_ids.shape[-1]:]
    text = tok.decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
    text = text.replace("output:", "").strip().splitlines()[0].strip()
    return text


In [7]:
data_dir = '../Datasets/entity_types/'
files = [f for f in os.listdir(data_dir) if f.endswith('.tsv')][1:]
print(files)

# Material de trabajo:
dfs_dict = {file_name.split(".")[0]: pd.read_csv(os.path.join(data_dir, file_name), sep="\t") for file_name in files}

['el_salvador.tsv', 'honduras.tsv', 'argentina.tsv', 'colombia.tsv', 'venezuela.tsv', 'guatemala.tsv', 'ecuador.tsv', 'panama.tsv', 'usa.tsv', 'nicaragua.tsv', 'paraguay.tsv', 'costa_rica.tsv', 'chile.tsv', 'mexico.tsv', 'republica_dominicana.tsv', 'peru.tsv']


In [8]:
for country_name, df in dfs_dict.items():
    df["superclases"] = [
        [
            elem.strip()
            for col in [inst, subc]
            if isinstance(col, str)
            for elem in col.split(",")
        ]
        for inst, subc in zip(df["instancia_de"], df["subclase_de"])
    ]
    df.drop(columns=["instancia_de", "subclase_de"], inplace=True)

In [91]:
for country_name, df in tqdm(dfs_dict.items(), desc="Procesando países"):
    print("País:", country_name)
    tipo_entidad_list = []

    entidades = df["entidad"].tolist()
    superclases_col = df["superclases"].tolist()

    for sc, ent in zip(superclases_col, entidades):
        # Robustez: aceptar list/tuple/str
        if isinstance(sc, str):
            sc = [t.strip() for t in sc.split(",") if t.strip()]

        if isinstance(sc, (list, tuple)) and len(sc) > 1:
            tipo_entidad = generate_label(ent, list(sc))
        elif isinstance(sc, (list, tuple)) and len(sc) == 1:
            tipo_entidad = sc[0]
        else:
            tipo_entidad = None

        tipo_entidad_list.append(tipo_entidad)

    df["tipo_entidad"] = tipo_entidad_list

    # (opcional) guardar por país:
    out_path = os.path.join(OUTPUT_DIR, f"{country_name}.csv")
    df[["entidad", "tipo_entidad"]].to_csv(out_path, index=False, encoding="utf-8")

Procesando países:   0%|          | 0/16 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


País: el_salvador


Procesando países:   6%|▋         | 1/16 [06:54<1:43:33, 414.26s/it]

País: honduras


Procesando países:  12%|█▎        | 2/16 [10:29<1:09:19, 297.13s/it]

...

País: colombia


Procesando países:  25%|██▌       | 4/16 [22:11<1:06:01, 330.13s/it]

País: venezuela


Procesando países:  31%|███▏      | 5/16 [28:04<1:02:01, 338.30s/it]

País: guatemala
